# 07 — Model Evaluation

**PrioMed-Rec | Sub-Task 7**

Full TRIPOD+AI evaluation on test set. Includes confusion matrix, Top-K recall, calibration.

In [ ]:
import sys, json
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
from src.models.train import train_all_models
from src.models.evaluate import compare_models, select_best_model, final_test_report, compute_metrics
from src.models.predict import load_model, load_metadata, load_thresholds

In [ ]:
results = train_all_models('../data/processed/model_ready.csv', seed=42)

## Validation comparison

In [ ]:
comparison = compare_models(results['models'], results['X_val'], results['y_val'])
comparison.set_index('model')[['accuracy','precision','recall','f1','roc_auc']]

## Final test evaluation (use once)

In [ ]:
best_name = select_best_model(comparison)
best_model = results['models'][best_name]
test_metrics = final_test_report(best_model, results['X_test'], results['y_test'], best_name)
print(json.dumps({k:v for k,v in test_metrics.items()
                   if k not in ('confusion_matrix','top_k_recall')}, indent=2))

## Confusion matrix

In [ ]:
import pandas as pd
cm = test_metrics['confusion_matrix']
pd.DataFrame(cm, index=['True 0','True 1','True 2'][:len(cm)],
                 columns=['Pred 0','Pred 1','Pred 2'][:len(cm)])

## Top-K recall

In [ ]:
topk = test_metrics.get('top_k_recall', {})
for k, v in topk.items():
    print(f'{k}: {v*100:.1f}%')

## Priority distribution on test set

In [ ]:
from src.models.predict import predict_batch, load_model, load_thresholds, load_metadata
model = load_model('../models/priomed_prioritization_model.pkl')
thresh = load_thresholds('../models/thresholds.json')
meta  = load_metadata('../models/metadata.json')
feature_cols = meta['feature_columns']
test_df = results['test_df']
pred_df = predict_batch(test_df[feature_cols], model=model, thresholds=thresh, metadata=meta)
print(pred_df['priority'].value_counts().to_string())